# ROGII alignment lab v1

## tl;dr

This diagnostic notebook tests the GR-alignment hypothesis at component level. It uses monotonic slope-constrained DTW on an absolute TVT grid, multi-scale locally normalized emissions, and both typewell and self-log references.

No `submission.csv` is created. The experiment is viable only if prefix-selected alignment improves the frozen Ridge by at least 1 ft pooled RMSE and improves it in every spatial evaluation fold.


## Context & Methods

The earlier experiment decoded a coarse offset around Ridge. This lab instead decodes an absolute monotonic TVT path. For every fixed emission family, drilling direction is selected only from a visible-prefix backtest; suffix TVT is used only after prediction for audit metrics.

### Key Assumptions

- Validation is pooled per point and grouped by complete well.
- Spatial clusters are evaluation slices, not random rows.
- Ridge is a frozen cross-fitted center imported from the geometry experiment.
- Self-log reference is built only from visible horizontal-well GR and `TVT_input`.
- Long GR gaps receive low emission weight rather than being treated as observed signal.


In [ ]:
RIDGE_CSV = 'well_id,ridge_c1,ridge_c2\r\n00bbac68,198.9741814829938,-5.691621278179624\r\n00e12e8b,181.67772282411448,-19.44980645137097\r\n01982c1d,122.54486244958596,35.80883502628973\r\n028d7b28,-316.25448792573854,92.46326606703477\r\n044af7d1,-122.13885027523742,-15.806662554601392\r\n0a57a29c,194.9881537331511,-5.620467516346534\r\n0d99f8e4,-183.30649226200057,8.845035972511653\r\n0dc835f3,168.15017071926928,51.08450193756605\r\n0dd99dc5,260.1263863261587,-12.88853764304027\r\n0e5e560d,123.0544062915661,11.69589555131276\r\n14a53cb3,169.79594466194632,13.089434904548568\r\n14ab73fb,-13.607913247547344,15.511736552568482\r\n14ad5efb,-272.15413468130765,-5.2532452787047035\r\n155887bb,144.72412640520588,-16.64915990704282\r\n198a5607,233.46101155409207,-6.615577660666174\r\n1a518997,-128.14469485963826,-21.244248880368932\r\n1aaf1da0,187.5322726533378,14.149844776247779\r\n1b08ed48,178.64649012975963,-16.51531941588083\r\n1d78281a,-185.71157096891943,21.049682250324775\r\n2043abc5,280.84359417872867,-27.817593218723715\r\n230eaaa3,-195.49407530270966,14.476535345988108\r\n25050f63,158.30633525595633,18.513690083871644\r\n25fd32b3,-73.72198479899353,15.524337877822234\r\n27300fd5,163.82092737680395,35.546044569183245\r\n28f4eda9,118.85983262559856,-4.70292974479552\r\n2b547943,-115.88642138170499,2.7966090740318075\r\n2bff06cf,364.80079555071075,-41.6525073085487\r\n2d196986,176.0674307134231,5.7019527372847785\r\n2e29e833,206.89256856854502,-19.515520344519988\r\n2f8e53c3,-63.720906590414025,-12.246398499842819\r\n33c40295,123.13068491328215,32.39451521995992\r\n347242c8,-159.34828111722445,-9.579904393519229\r\n37344c2a,-90.83954243156651,-21.232473924942557\r\n373ef48b,-119.26675738216196,0.3290973198079312\r\n37d36812,96.84582446430056,16.2401747451803\r\n3a86fe8d,-114.80732130053522,-31.229269040846663\r\n3aaa2b30,-182.2946686605379,-2.7286643289834966\r\n3bbe1f5d,-175.45043982834466,-47.38855608874971\r\n3c3dbcbc,101.24860319419744,31.252187453176113\r\n4121c517,151.018524460229,8.369513905935591\r\n42669188,169.0677229198708,2.0621169183712187\r\n43e16325,-69.95870465832994,4.38769429176796\r\n44441e54,-118.13390816390006,1.47708715795957\r\n4463446c,-173.55624367482815,23.727940074564348\r\n463c38f2,167.36411676200376,15.871696471767564\r\n46a7190b,-79.65894219073681,-3.2209478810315857\r\n46dfcfca,-78.5290182208789,-0.25784245132861106\r\n47222616,96.0535417618295,18.008806034233864\r\n4936af16,-101.84448780163268,-23.04382974257488\r\n493b5b31,-108.11471913494685,6.542516181453457\r\n4c3df468,-239.98964331438646,-4.761991093684669\r\n4ddde4c1,148.8100330477582,9.344841341354616\r\n504c8b08,180.38064013340224,1.4165918986944508\r\n516088b7,271.22854641809545,-30.35995084531983\r\n53134831,-105.92182864928694,-15.093207763061528\r\n543198e8,310.7060145480494,2.718070944237743\r\n55efee7f,264.19790899489647,-55.371106652182235\r\n5663b2b7,55.834018049722985,81.83418900262811\r\n5693dde2,-139.0490316221471,-35.49623794558783\r\n5a5d1982,-122.66226133822042,-18.18838118595124\r\n5aef5c6c,-99.70469693632415,28.655522223894238\r\n5fb1c15f,97.17457525456214,40.50589511762165\r\n633774dc,187.2950187774875,17.700813190646475\r\n660d9546,-61.91075830807673,-55.03783439747775\r\n684a6fc1,-291.7985103569517,40.6447429500413\r\n69c1bdad,31.699267998891063,52.44738918850804\r\n69e41fa6,-145.96615715589184,-2.5387457657525005\r\n6ea22614,-201.33264536818987,-9.40076839438263\r\n7256229b,-214.54361265546677,5.0558130818389175\r\n75c20ec7,169.3401363411579,12.92208490746351\r\n764072a5,68.65467940726089,25.248573232662856\r\n77b0d905,100.15879481974073,20.2133879840659\r\n7987f2f2,-172.06625040003627,-11.01370334457549\r\n7bb17b96,-167.53640706657657,5.8438944024796795\r\n7c607683,201.50472694620908,-20.3216709617144\r\n7ff89f8f,194.4234041827093,-12.308930419793139\r\n833af382,42.76514877291901,45.077487569064644\r\n85380836,157.76664476571827,15.816772363647829\r\n8648abae,-185.71346506884785,9.25265334674587\r\n865033f8,100.07723373969219,21.680629006579075\r\n8b5be31b,266.3661127912331,-26.19115746090288\r\n8b95d6d1,-39.359915984356185,-4.258301783514698\r\n8bb9c1e6,168.70623315204088,-13.738402107357963\r\n8c8348e8,-92.06307158844089,18.679475024043924\r\n9283ae69,156.34018674826643,0.3984628249541968\r\n9298ad5b,104.0600803816993,26.82131920887352\r\n944f36b9,-153.84420481242904,-19.942078937938852\r\n9719aa04,169.32844741680253,12.741876077299679\r\n97cd5bf9,114.61250657022055,11.867313235434885\r\n9896fc0b,-98.35998808661748,-15.601130569854355\r\n992c99e8,179.6663256034069,-22.669387119578335\r\n9ab94eeb,198.16265670352266,-17.951311249921684\r\n9b5b61c1,-148.6466491185463,-17.56710296418353\r\n9d4272e3,153.6568673753928,-25.058050265996584\r\n9dede370,225.30625892072004,-1.4937298256813842\r\n9dfff011,111.84436816291145,-11.345081835307845\r\na0b83014,-125.54313265213409,0.9736685631887969\r\na3518960,158.09758004283879,-1.1297148305206073\r\na4719920,-194.40841028127468,-10.397795253682027\r\na48640d9,-167.65939967655774,43.81130241421024\r\na645da9a,-224.24511166794514,9.792780201881996\r\na692b17e,70.01135483733003,46.42877245148552\r\na6b8ac67,286.477243063833,-52.52974743659261\r\na85e4bc3,-136.34607914196184,-64.54034375044266\r\na959858c,-156.84113276252145,-31.15598039099457\r\nab112491,117.37381073404185,17.41854605021333\r\nab18d3fc,-110.39564714928105,-46.047446157536434\r\nae35f33a,-152.24534136208177,5.337246320404117\r\naff4e561,162.89423108864602,-32.9139020897143\r\nb19b0395,-115.27815262165726,-2.306376798754142\r\nb38e3116,168.35180059287788,-7.296414553329402\r\nb7974e66,121.6103952097684,26.039143243550466\r\nb8c49c1a,-137.33894568046054,4.7026651688956305\r\nb95e7121,-286.1330882437724,-5.026287490663043\r\nba48188d,179.2429255259235,35.44824874957244\r\nbd4ae5da,196.61050802226137,-6.518432018252197\r\nbd8f847e,-189.82815093983714,-13.901966879239826\r\nc07fd0cd,-192.47388634796135,53.85506708759288\r\nc3957531,77.63343963260388,18.274414263561823\r\nc472c0b5,158.7754827365082,7.789956132130164\r\nc50b42f6,-140.4492908303735,3.1228737677221865\r\nc593337f,204.21114589841144,-22.166829091426028\r\nc84c65be,306.12448047482667,-20.573723304266345\r\nc9578d27,172.4563523741904,-4.341397902650444\r\nc96c018a,-181.81158334848587,-28.835983823795832\r\nc99f9fae,-262.80651667336696,-8.826779948138892\r\nc9fedcf3,-90.07410232575614,-18.779386314847212\r\ncbe51de1,-198.10807251692404,25.659938495008547\r\ncc08aa63,87.68111462950952,22.095866691617072\r\ncda7eb37,115.7124932978281,51.99058212974137\r\nce8399b7,-106.60810380436726,32.07320345944867\r\nd3df146b,-139.04106218469963,-29.99776737604354\r\nd40f61d5,-142.7781561529645,18.740529176289517\r\nd50c649a,228.83394237429425,-11.941178525934056\r\nd7ba4f9d,-313.14295542622784,74.09952528971466\r\nd84b621b,166.18669170585378,-37.20687626124585\r\nda2d4d7c,-88.71640375059738,8.299913559962707\r\ndd7d638e,145.57164758901152,-7.430997149868297\r\ne46f4ef4,-165.18444189395285,-0.7946803237168583\r\nea41324e,-134.68488463047325,-28.60459367399492\r\neabc711b,166.10713762920676,-30.79651676917515\r\neba6605e,-8.630278662590811,-31.635627449830544\r\nee0300f7,-117.0053194538462,4.569407666073116\r\needa36f5,-150.3988872875045,-19.592420120460556\r\neeed308d,-146.70617637019393,6.587278422700851\r\nf021b650,173.90914582050902,12.771999831825617\r\nf4d12d23,91.74387618567135,5.096018116785511\r\nf6bc699b,-114.84720787442293,-3.6999213970992613\r\nfb03ae90,52.975214440489786,69.2940050899036\r\nfb3848a1,220.32699076634424,-22.555736055152682\r\nfb73b13a,294.1861003428124,66.66601476075164\r\nfc0d20b2,168.5313948051438,21.85246709370011\r\nfd3b4faa,180.66625234514342,-7.631249461537225\r\nfef8af96,-63.085113208614736,-22.746024037565764\r\nff8bb73a,290.2139688372336,-33.39483316983398\r\n015fe0d2,-164.80183398380962,-5.105202220371857\r\n02e7fe5a,137.04193941022382,-38.08336728179756\r\n0849b4e2,-136.54298519129836,-1.4329317740884713\r\n09053135,-201.72992562665942,26.51829750239647\r\n0955cd6c,-100.10923782803401,-31.298005811069306\r\n0be01c24,-136.5519766342184,-20.181294507475815\r\n10a1281a,229.1091471775654,-30.60714675532688\r\n1131525f,-109.99344279812897,-10.964624056324336\r\n11d0f5ac,155.30209362877753,-6.080149749839831\r\n12203f2a,161.8488947528881,-24.775793148075092\r\n122bd617,212.07286922766596,-27.78051688604562\r\n13ce113d,135.0938594556719,-33.275028679634474\r\n13f598d9,262.7046471067573,-33.539231776268636\r\n19871e7f,181.91066757343947,-5.267306510740652\r\n19fb4f7b,179.46969675410762,15.690019881304792\r\n1b1eba53,-62.37480510493765,-32.76120010897868\r\n1fdbba44,244.94793193219104,-11.81479579401759\r\n201661b5,135.55428268384057,45.61939375717539\r\n204cc64b,165.33870276016148,9.035254533502371\r\n206b6193,246.27306257174095,-2.3843823718697834\r\n24d8997e,105.14082387765302,8.48620141348771\r\n25939962,81.42163742858776,43.90619879133814\r\n264feceb,259.15713108624783,-52.08615571603606\r\n2ab29395,-51.16527330670522,-12.377106999588252\r\n2ac7cb6b,204.94090017986358,33.65212029862449\r\n2f19d536,112.3060973194334,-0.1261930054559659\r\n2fe023be,131.39600046194204,49.498160240704976\r\n30d75030,163.84312979131826,3.69028805062122\r\n353e5502,218.84066222330063,-5.6895987194709114\r\n367456ce,-201.10506078077856,-16.920474914959772\r\n368131f9,158.20594299164526,31.98946310553227\r\n368fe682,183.34266409810854,-20.650493846049216\r\n3ac16ad0,-175.31255515114802,25.423260892747653\r\n3b21ea64,258.7011251614502,-31.920909221329335\r\n3e011332,61.05205677476077,55.138942238381546\r\n3e678ea7,-162.03111319212022,-5.109094505271152\r\n3ef6372a,-140.4303288786755,7.153675232113905\r\n42c538a1,-56.4273816233318,66.29215740014028\r\n43c7f85a,-180.02686371347235,-25.780714993479517\r\n4654e7de,-116.30793262085601,-54.217008708664544\r\n4a335117,-165.2970343219948,1.7703500133030319\r\n4cd1ac61,203.96584097235507,25.452626694338505\r\n4f4afcc6,-162.70350020607245,-26.956833061806677\r\n521a7819,272.8505026833332,-68.08964219335661\r\n5254b6db,95.37058402834195,19.672822359631482\r\n5542c301,168.3527532900992,2.0573953323684258\r\n58850d2c,72.63094154309296,4.952664051188754\r\n58fa8486,161.57469411701965,0.9825748359903008\r\n5aa403d0,135.78116546379474,10.24727054053076\r\n5d11dfb5,201.01450903788313,15.287296513212041\r\n5f6756c5,218.80323824929846,12.411312946145113\r\n62a6f539,131.860796138374,-1.5291111471355099\r\n65156b96,-151.74869140581285,-20.852139421873968\r\n67a8da4a,172.35939241648094,-11.387907928188994\r\n6d1d74e1,-229.93730064626382,-10.205678832891259\r\n708caea9,356.32224972413957,19.350624391026575\r\n70925e23,147.8100120902315,-46.6796365330058\r\n726f3dc2,251.49803823292396,-36.01195681001176\r\n729665c0,-113.44553669534716,-36.75014681332152\r\n72dd8501,149.95803456113813,-3.857499809887197\r\n739a914f,-165.91658856926907,-16.032247494350845\r\n76201745,-158.7097659789729,1.2693109753899596\r\n78a4a386,189.02729020603977,-19.619125330170455\r\n7b38844c,-77.0226527939932,-42.43508813138104\r\n807f0298,256.4354539626136,-18.00583444332281\r\n81bf5923,34.595522206644816,23.311944597897984\r\n82fcf37c,155.82815052663474,-37.54510488586961\r\n8382f4b6,190.4536366310479,-24.600853507760416\r\n8478df29,216.3833438648904,3.154596227409195\r\n84815d5a,88.8346560360424,28.74984499363627\r\n84c3b497,131.75079305785934,26.460097766779555\r\n86454a6f,-338.21737380570204,40.52518510648468\r\n87695469,272.33829180799495,-35.90277172909175\r\n8902c3f6,-180.93201365898994,39.050660397814966\r\n896d15b9,36.737698738075125,88.67639487445459\r\n89f36adf,-95.06766684972496,-26.476272526330177\r\n8ac2f237,145.83169923992534,9.868177085091018\r\n8b12bab6,122.42542521979709,10.931339921726053\r\n8bfa881d,84.19301848619823,8.259974521440478\r\n8d5d46d7,221.01049708961347,-22.675155943181633\r\n8f201368,136.87972325967567,34.84259536004\r\n91db7070,-146.39147515807932,-24.847237791176962\r\n939d9c34,229.80180267165258,-61.79763683970176\r\n992ce078,53.41465145376799,46.488635098745306\r\n99529c45,147.6525221070098,1.9167021361492325\r\n999daf80,124.36312446481105,-0.5905826679558769\r\n9c350a6a,-180.26217778321606,41.74968228032636\r\n9d5e14d4,255.0564045174505,-31.2533878141744\r\n9e3e364d,-173.22929780246233,-0.028660312808963306\r\na08d9e63,-166.13006372720744,25.84594565964044\r\na2e8e7f6,-102.58320252687395,-32.85213787392182\r\na4cdbb0f,165.9958068906066,12.520468124459855\r\na5aa7973,210.61438721786854,-7.4102388551197285\r\na622ce4d,-263.93702046010765,-16.915513128444566\r\na76db406,-153.73064773358743,25.609059436397168\r\na7744f5b,-152.7313562272029,-2.3616568616841027\r\na7fc3e6b,-179.18094909480547,-14.887509121631517\r\na9c9b150,159.99529838235026,3.5386468419989816\r\naaaf3c03,16.957847313215268,79.3158711032429\r\nab94878e,307.57846134301275,-42.719049488494754\r\nae0784a8,-275.2057906894258,19.56934381671742\r\naee6393a,-143.67240705748193,-6.096632476013564\r\nb04b58a3,-163.4981349371487,29.94359628659216\r\nb3388334,156.51065805992272,28.83072057434578\r\nb37fd114,-304.6560936976883,55.397148433599014\r\nb4d8dd4f,138.39447446696104,-10.813882100814128\r\nb4f37e6e,-83.63386497896015,5.144827394334808\r\nb7cdcecb,-225.3733950803284,-22.621608111624266\r\nb7fea9ef,-133.07757610853508,-28.875688555617753\r\nbb2cdd83,176.62360534853627,-36.203792848002756\r\nbc4381e2,187.97355260260903,-42.91316408005608\r\nbd2c2f34,271.646030077495,-25.720589034781614\r\nbd6f0e19,-127.91070352162535,-30.973528903402755\r\nbe35c8f2,168.33358057837432,-17.758965169958486\r\nbffe3082,-182.31750122586928,-8.26215886240249\r\nc1d046f4,-182.12489111079947,4.724939847806384\r\nc43812e2,240.57752288822704,-8.998122880183791\r\nc9e6734f,-110.88630291168866,7.93008995242074\r\nc9e980e8,-142.89271613120187,-13.415719337532066\r\ncd7f1687,179.42104272046544,-13.610124559835484\r\nce55ba43,115.0856512757633,18.771171922475062\r\ncf50c9d1,203.85088784198263,5.780830660596802\r\nd00e7eb9,-127.37077517190852,-43.49593804422259\r\nd07aed8f,-69.24555039095176,16.664999267929065\r\nd085e611,-243.01697179408458,12.009644675732392\r\nd0a0e7b5,122.93614035165176,-13.328051521786765\r\nd0a43760,-99.78684080131066,34.12687131716735\r\nd24ff243,-132.97491627299175,3.1978190943934317\r\nd6c320d9,-108.54650387792199,-64.73346336695886\r\nd90aa14c,292.882526220932,11.012732238744736\r\nd9eca87b,212.78846857397787,-31.695934032546873\r\nda3bfe4e,-78.79423342440192,-33.938039645416644\r\nda3cf798,231.90249884448258,-3.032523414964852\r\ndbee6c86,288.82549751901996,-55.834288471163255\r\ndc5fbe29,-161.45336707861765,14.017683440443015\r\ndc7da28f,-143.66855625740848,15.368698427764304\r\ndc7f9757,83.5416031581361,39.735722964471606\r\ndd6bc5b2,-156.11710004681444,34.47576157557779\r\ndf73b8f3,-172.10954262754535,-15.120787814969326\r\ndf8290be,63.09713337196125,55.02279864850462\r\ne149204f,-202.47889996365035,-15.97997120809358\r\ne25f1537,172.34171470858107,-2.6917192983819627\r\ne6c55748,-139.97709180027863,-3.89980629437522\r\ne7d0c1de,-129.4144968899637,-23.623815327194215\r\ne97032ad,112.53927926109671,14.036124941304925\r\neb934281,155.7929902596183,-5.475380501732396\r\ned6436d6,79.33381570756465,14.128565018289578\r\ned6e6e54,154.53701207656874,15.387176242794077\r\nedf885a3,99.82338567953677,9.947474130497682\r\nf88ddb26,25.035420759929515,-108.88305834426973\r\nfa667be2,274.8908612641986,-28.136989455421517\r\nfae0c593,134.42572825948088,37.96468266460426\r\nfbd68d27,-134.71423834437203,-56.35575823972059\r\nfc03f21f,196.24743871776533,23.325856734412433\r\nfcfcc902,-74.20743876676978,-5.08175951166551\r\n000d7d20,109.90946572551235,0.8159522544854818\r\n0390d174,-170.1593917142725,8.815510411189194\r\n0498acab,-134.89706588180587,-9.212873930976425\r\n052d64df,93.6775206160028,-8.486307414704132\r\n071d7b45,136.85796625626847,8.94091040408327\r\n084d66f9,-126.22754482375689,7.562977638029594\r\n09ec2ca9,137.16647134123073,24.64240617887592\r\n0dc5e64d,-227.918792745361,21.829565751132385\r\n10b89021,215.4984940083473,-73.76982483979624\r\n10be2420,153.43734594559206,10.735745193591164\r\n113011eb,-113.47397056520086,-12.599712090273947\r\n125be846,284.50119430888327,-50.481903119487946\r\n1285a37b,189.6601968097822,-6.299464818023111\r\n1295a25b,-154.14536938544668,-31.26734819724573\r\n1550491d,220.50378605564353,-39.44141005609836\r\n16e4a047,104.16687546641694,18.579692075100596\r\n173ed841,161.2398007976802,-0.16556076872295877\r\n18d24f7b,185.23896131789044,-7.160079804479192\r\n19137a89,-224.10545306298022,7.876505161970554\r\n197f8a5a,197.07179235372968,-20.154953572096744\r\n1a730ac2,-147.442371602402,-0.08993091444399903\r\n1b1c5372,-135.73636700462686,-12.631196410104861\r\n1b6ba517,242.3488871654264,-15.419347891495272\r\n1beca81d,-105.84424428577861,-27.697403791612103\r\n1e30dedf,275.7291349430257,-19.20700917346465\r\n20096916,135.67073352807802,23.848012690333146\r\n22c5f93f,137.8085849721253,19.54354202942384\r\n2364716c,-161.8791893815086,-14.451137915516115\r\n27c3155b,-59.293791595476684,-39.39348033169162\r\n27ebb9b9,107.95296781991777,68.73620330374098\r\n283269ac,86.43724307267067,13.07181847834963\r\n28473855,164.64177179944264,-33.78244721277264\r\n2b034622,190.80802570434858,-9.815089331558623\r\n2b06ad65,128.53048943284963,-2.6951016105841816\r\n2cee0cba,-185.57617635905132,4.063606333872524\r\n2d2d0c6b,-90.36363183766684,-11.752573683754646\r\n2ddad940,-124.53900420892631,-54.368231369926384\r\n2e63d9de,-124.53677749609588,28.081390177831924\r\n2fd68f7b,-269.24511122867773,49.327819572988794\r\n3584a6eb,129.96677575916226,10.090547688245714\r\n3690a146,-110.21985877127685,-2.0945768075004043\r\n374be387,309.8487960149456,-51.29027718414524\r\n39b51819,152.1243048996572,7.904248680394383\r\n3a0af893,110.64708279894049,6.495009525822031\r\n3d96d897,-122.53489470603682,-70.66687107198528\r\n3f603129,-209.71967807308317,17.989345958630565\r\n45fb3e21,170.5204917175541,-17.37613821447709\r\n46d24a09,200.97428501697908,6.4701668515949855\r\n49dbc14c,-125.35318991038514,-0.8954930206777352\r\n4caa7289,158.28522253962333,2.766349844126054\r\n4cecf7b3,183.04530847200905,11.765851441511659\r\n4f69febf,117.47015817065761,21.635971722577008\r\n511e1db0,-190.3920290999658,1.3136224552518794\r\n512943b2,-126.48139510914314,-19.68825622255827\r\n54a7e3c2,207.2070353622044,8.456762440994924\r\n56b00794,-208.940372573187,56.53095937681051\r\n57796579,251.38633128545482,-27.768018753973273\r\n591cc951,232.47142790272127,-17.36139643245637\r\n5a63b5d1,100.072249960389,4.7928709889360945\r\n5cb483ac,-126.56253236681926,3.6610895019616776\r\n5eae34a8,-138.6368236078236,-2.930135567645877\r\n61f27424,206.61265693100847,-14.814849114651807\r\n63559d57,-181.82583693839842,-2.4753217254411912\r\n647f2a41,-139.10204809412875,-2.341733129958844\r\n656a067b,143.793835990698,-0.14772518466522988\r\n65a5466a,-138.3871174682074,-37.99364040793405\r\n6767c111,340.30016912040946,-3.376218547085066\r\n67847288,323.4459930709169,-68.79474503088001\r\n699ea575,72.44378244825415,28.18026861110326\r\n6a0ff78e,-166.72787605315074,-26.412239557570416\r\n6d590e26,269.4989432855468,-33.210586213255404\r\n6d6d93af,-124.16417560464677,-5.836660196782399\r\n6dbe4b60,-151.9411147076911,-9.46486835238121\r\n6ede926a,169.53117782993718,-20.176129663996697\r\n729f4217,-179.95259579843687,-14.22518163560191\r\n75d20a82,60.0410883848577,70.17426928672356\r\n7993a768,126.54886184338594,-21.985905811419997\r\n7cbbd047,-250.2001305685193,8.36601776755913\r\n7cd4bb31,-181.0497665751668,-32.44132396983236\r\n7d57c75d,-155.53940922662542,22.548631157536498\r\n7e721392,114.40740232605745,4.123913736847273\r\n81fc01c6,243.79555199120904,-27.00272951705653\r\n8612b37f,165.67972811679968,33.54362245922125\r\n884ecb5f,-187.2540832019357,-16.70586132179632\r\n8995c945,309.568927272122,44.01282093105699\r\n89fdb2f2,-39.42993861501375,-21.5877940975374\r\n8a89da23,-107.88536134896901,-8.334925201971613\r\n8a9c8932,197.01152494779546,-48.390980269788024\r\n8c167025,-215.63379777922216,63.68372466971812\r\n8cc21f01,-108.20600261187954,-5.177009147109947\r\n8d92cb49,118.27738264195673,12.465872030756982\r\n90386ee4,-153.01389860195096,18.496157491884496\r\n91abc4c7,172.6550959420706,12.577565041039772\r\n93f5d2e6,-106.76153797964497,11.085511353700007\r\n94d813a4,222.23968615446105,-21.56988066655373\r\n96936c22,75.09353490157321,21.239654007182295\r\n96ae5806,-323.4722225285348,8.706096200397111\r\n96b7eea1,187.16519257279566,6.712418291108504\r\n995ff498,165.73951195538,-11.89915993646426\r\n9bd2bba3,260.1587868018952,-39.36433951087624\r\n9c211c53,289.56220055562335,-25.132978560117838\r\n9d072e4c,-134.93506264536248,-8.563067907669037\r\n9da441ba,-274.2845236137742,13.285385968151447\r\n9ffce529,162.3923164003129,30.759402554222795\r\na0383629,-146.36745594984433,-28.08445947010384\r\na247e7cf,-185.42353797362094,10.554915579300328\r\na4f989c2,-163.09719684481598,0.6064524646221212\r\na5e6f7dc,-286.162275483031,32.50469014266465\r\na6ed2eff,169.12214444762293,-24.69280594416287\r\na783cc24,-56.21154215783819,19.806199310606925\r\nac6f01d5,-112.0269897555902,-17.06335533028757\r\nad2133a8,184.49836270416077,-21.52876637419116\r\naf7a59ce,-98.57433866138834,8.679991471759605\r\nb0f53bf4,118.06252854202975,-14.186954708572602\r\nb1efb8a4,183.56367678026874,-21.88547473472999\r\nb85f7b06,-152.0541847920745,-11.94435285812519\r\nb977be4a,-211.9986865889989,-48.46655434391718\r\nbb337fa0,212.93236048716867,11.251525134823815\r\nbb682ebd,169.17325740316227,-1.2893530159287097\r\nc1708b88,-216.32223512127996,-38.4009241503442\r\nc2c6fc71,-209.98338822128412,6.659691783098442\r\nc47e0767,115.37182274911328,-5.985449172173595\r\nc6c96179,-131.4093627038185,1.033834525950772\r\ncb34231f,-99.69848751394488,-38.67861635250611\r\ncb879f40,271.31821284888815,-21.48261935422898\r\nccedb12b,174.76156404629216,-12.893494952355795\r\nd1ecf309,176.8225200591593,5.064878555824286\r\nd1ee630d,-273.1036402853091,-11.91283172523982\r\nd273674e,115.83641251410143,-5.114918458572998\r\nd2f3b1ab,-159.93054169779492,-2.6179336622657368\r\nd463e527,149.01576184842443,0.40303923501923045\r\nd4f60a28,-162.0827937089392,18.446031130011104\r\nd9ae59b2,-187.5922179232019,35.02087336004904\r\nd9d6d94d,187.91885268412378,-23.749574846259044\r\nda940160,128.8116123359915,-3.4454125685914097\r\ndafc88c3,-128.13820713019166,-3.103511866743263\r\ndd168de0,157.2014898994513,-19.840375732343304\r\ne03b45fd,302.7604985226663,-17.20809978684705\r\ne2fc7745,156.95060615268335,-11.235732691562816\r\ne3b34f5b,-155.2793299170462,-12.803633772856312\r\ne4726608,155.20052738299304,-11.533901599084615\r\ne9949a23,157.1304008880325,-25.440934615388986\r\neafb863e,64.41761845578822,2.4146853813656457\r\nec13af42,-128.58538443333836,-3.3535496520732515\r\necdaf714,323.30345576372736,-84.24191236664412\r\nee288beb,-221.48385910878346,44.344406276676985\r\nf0188a48,-246.56871551351438,42.32037722868501\r\nf03f10fe,-116.47821248141014,-2.924930364312265\r\nf074d277,106.12904340448878,10.748744721480463\r\nf321a31c,-271.75417848889265,65.53394559465525\r\nf439a621,144.12177635386567,13.842279124314915\r\nf5859199,270.10984371396285,-4.989416116717678\r\nf8662c22,-130.943275237101,-46.78517500259864\r\nfd710aea,108.37629816468308,-3.6202633145514236\r\nfebb4411,-140.78486527697507,23.411320349519915\r\n01869cd4,-201.62818614393973,-1.1041680154618243\r\n03a935ae,166.11374903315556,-66.8751864743343\r\n059c8f24,-339.80199760964814,107.35897996081135\r\n05a0ee4d,210.20025583347586,-14.58842394344667\r\n060ab2b8,213.96009558998475,-34.688174398988835\r\n0c1b160c,-159.2181382057359,3.269478626914898\r\n109d9d70,194.00870225274235,-31.377807776100333\r\n1590af81,167.50993710582088,-18.24902651549138\r\n1770d7c6,250.91195848992407,3.929279145212483\r\n177622d7,114.15548785476882,-3.704829796682035\r\n1f9afacb,152.41375521127338,3.8978999853777934\r\n1fee2b62,131.58763951646236,5.997668863706506\r\n200d233d,-135.98835414185743,16.704325926080216\r\n2075c696,-122.8388590174206,-13.569012846688436\r\n23b9beb0,-234.38837755703992,-6.574224967729963\r\n261785ee,174.5684086601472,-1.3628028053558223\r\n272abef3,-117.26557876711452,-27.365395702000338\r\n276b012a,-216.62991101763055,82.02584438810368\r\n2d179ded,186.90457616867786,-23.53220629223065\r\n2d35f86d,167.08535124159783,24.31976632442076\r\n2e43c2ea,157.40954933332108,22.400698854531097\r\n2ee0235b,236.74009653922104,-38.4178454137966\r\n32fe84c9,-98.39069955398483,-45.52116112913135\r\n389ae58f,-143.7608984808678,-31.87681868418304\r\n398dce4b,135.78080066769377,-2.0139250769006383\r\n3a223519,-145.1529803393106,2.766512187539756\r\n3a824aa7,104.48257244083315,17.756400533454567\r\n3bfee975,-135.6634081182384,19.54846982527894\r\n41fb0192,-96.46429552114586,-27.21085623311173\r\n4630ab92,110.0555283145987,7.105841160172561\r\n46a301e9,-248.18920746037213,8.664760317369936\r\n481de5fc,135.32602771675707,55.29925886501727\r\n48c1f673,135.6742290687502,24.204380461256743\r\n4a035ec2,-68.26349014127591,-21.14312943146375\r\n4b462989,170.39013129899894,-12.000964501658627\r\n4b831d20,248.29307198056694,20.425028773957855\r\n4c3168cf,124.4376552827819,6.876996385526596\r\n4ed93db6,134.73323620930745,-9.56218090514444\r\n4f3eb9e9,107.90155096453644,-7.783492707896239\r\n4f4ac5ce,150.64346745209474,-5.465049759411672\r\n4fbe9c06,-153.95324441080498,-43.84157094424744\r\n529e88ca,158.9611568389983,63.352388345975704\r\n52f1e77a,-59.669099199971285,-57.14783489385544\r\n5305524b,185.04045251905927,2.6835829650606513\r\n53f23031,-282.1157806648869,8.589809434195606\r\n577eec7c,-168.56348189707063,-6.118517317067303\r\n57f05c51,-144.9235478320037,38.517796844357704\r\n5a1a8fd8,-141.79475601115604,1.4607726745929241\r\n5a5269a9,-243.2097114151171,-50.62245042816726\r\n5aa03df7,-138.03928527823376,-29.054200859577527\r\n5b7b6324,209.62228378518614,-7.834227094095883\r\n5bd25f59,208.96847808293853,25.2141755018275\r\n5d7198fd,178.97027473005977,0.7480751513320758\r\n5def1ce5,-158.0233844391679,-50.49418899702498\r\n5f4d2a52,74.37780487106221,83.74286831771084\r\n653612bc,-53.227953641945774,-54.16287811557428\r\n698930d1,186.4474552169686,-24.503596411014083\r\n6a8fa194,38.10888717063162,66.3172186590248\r\n6ae68655,189.33681097275775,-14.266498948759319\r\n70e1788b,286.2931715937679,-58.94809692036143\r\n71ccf778,40.79557919544994,44.86635192357322\r\n7224331b,258.0589433332551,-22.307009088580404\r\n727a3a10,-0.2541833184319202,19.080456216170827\r\n73348914,-166.8188325290928,8.076886532227958\r\n74b6186b,131.2857161769768,10.057763117500658\r\n77e4821c,-167.6058492161036,17.149383714862186\r\n7850c72e,-143.0875730880373,-30.515912255350297\r\n796dbd4c,-180.2391233553048,-45.39618268684009\r\n7a5660b0,132.76019908248298,-2.6248506233505866\r\n7d50373e,-97.05774116207857,-12.826506948993469\r\n7e208414,-112.28182008944732,-12.370423110528323\r\n8050c789,124.68257799296829,-14.472595407893728\r\n84633df4,188.3500790185788,19.8281090278402\r\n8a3da6d1,103.43729352828231,75.56275758821485\r\n8b9d8326,-174.12306269241986,-4.301171633742743\r\n925a7cd1,173.83034805254027,2.3244364486273517\r\n9314ff13,48.88581583902935,46.163870565468955\r\n940a48d9,-96.35338970586655,-9.662901245770144\r\n9426ec1e,-59.92472252011662,-25.39976826804097\r\n94467f50,75.81019532342185,57.679015648841\r\n974802e7,281.2331365803329,-73.66616341117643\r\n98bc6f66,187.5046634100268,-16.659063419658978\r\n9a8ae0d6,165.3116690779679,8.59785380572618\r\n9ad5410a,196.33925585444325,1.770962205664051\r\n9ae248be,290.73429047481727,-38.801111255672254\r\n9af1ffd0,159.2754441486182,2.088889218310795\r\n9b9e3b49,180.2703307491467,15.404168747749338\r\n9d3ec64c,-111.00646139251621,-30.25834897480698\r\n9df1b1f9,171.21692582925937,14.652776676811563\r\n9efc812e,-121.74881590831627,65.77049902587069\r\n9f0c7bae,-58.24345994594135,-19.34579461919377\r\na0e92ed6,130.43206736224403,26.45411839171191\r\na6f967fb,-83.44476494036718,-23.660988442928225\r\na85bb86f,84.3143240948149,40.43807284167656\r\nab6fe95d,174.5011182802176,0.7856561600836294\r\nabf460d2,81.8591749682256,11.344460936213068\r\nadd9c322,142.62793834660192,-5.304349639768341\r\nae069086,245.384988051932,-30.996406762427863\r\nae8959c3,219.82867980550566,9.053949417363482\r\naed44918,90.94535647876877,4.851530451932835\r\nb0d42b0d,194.32862713254752,9.17331188722481\r\nb56bc0aa,197.25386751931586,0.543313113060444\r\nb585b4fa,154.2445457304499,14.765420127008724\r\nb881d27e,-156.38992861072518,11.029725507986932\r\nbe063506,205.62189095947684,-23.26367449579676\r\nc03b9305,200.06417305149608,-29.617933524030096\r\nc03fba65,120.30250862172616,-0.006642395037587789\r\nc2c4db09,-242.99940930276955,-40.44540868836826\r\nc36625df,160.31523752731619,14.181675545478797\r\nc66be2b8,195.19700854116115,-10.373777363560622\r\nc6797dd7,-147.36421290353093,20.91516996394924\r\nc70c0e01,-139.48477991013254,-17.100730334025734\r\nc8d9680c,221.2615906497963,4.677466261178912\r\nc908edd0,112.53152911129682,28.691722859759484\r\ncbb406b5,187.0235558755365,6.510429498455332\r\ncbe62450,198.3640729139217,27.324728389669655\r\ncc8a1d57,228.20539755872807,-29.31456676846504\r\ncdc60a53,208.81586172508213,-13.550663475388822\r\nd011f41b,-42.048129108097456,-28.61634286623735\r\nd0cb63ae,291.0744590976601,20.909047217751954\r\nd114d9c2,177.41538607406892,-39.48599100474851\r\nd1457cc5,199.05084341169385,-42.84922325594198\r\nd217e0c5,-113.43051071691455,-18.14164925370533\r\nd60430e6,98.10186368895462,-6.127004082367014\r\nd63196fe,138.85243299953055,17.264834415155295\r\nd924e971,143.05928432118708,-13.782148114374968\r\ndd64382d,135.01998848490842,-7.823178398822428\r\ndf71d1ff,-199.21215170312547,-46.62982155416539\r\ne0079ed1,-126.81195708372728,-8.627612314797695\r\ne14d641e,70.64044418073601,38.188555152042355\r\ne201fd6d,140.85356199004764,13.07291964539413\r\ne5864244,-139.19961154301234,-18.098408220384027\r\ne5c92e59,191.59073495595982,-1.0828921575904262\r\ne748dd88,-118.79602261735907,2.45414613288248\r\ne7818f7a,-220.22576745728077,-18.110076566756582\r\ne7c36e81,132.04222030984963,0.21692477540377064\r\nea0b99b7,-287.821080086643,-17.97006106198004\r\neac10e86,-96.27343751663108,-37.96504641017628\r\neb06a4e7,188.1355503651318,-22.305907835548187\r\neb640a35,-186.5108826484685,-22.9482718239182\r\necdab904,227.44526106121322,-12.954452821427685\r\nef8e3ed0,129.02795233189767,12.927562496821004\r\nf07d7037,130.27832812168776,-1.29161113627814\r\nf2d4c8c9,181.12836469523813,-25.265935936896327\r\nf40cd091,199.54488827712947,15.056708141330915\r\nf49fdea3,227.72554049276505,-11.152058961488663\r\nf6d009f4,346.59899813036157,15.074727119908406\r\nf72b268f,-244.79163596980953,20.26997174276589\r\nf8f59188,166.1343958775537,-1.383456918358109\r\nfa16d114,-124.80849051333537,-19.157216699012253\r\nfa31da94,-220.59621020496098,-1.6312318275022704\r\nfba7683c,11.331600196720883,31.40902687252892\r\nfdfd57da,-195.1394787316343,12.04471312418596\r\nff0aea78,20.251300068849744,58.41604363238401\r\n05948241,159.01081495610754,3.3651303208116614\r\n06df5958,-105.39952867763435,-21.840082940883885\r\n08fecf00,-117.76884753421231,-26.381044138651426\r\n09441b8d,199.97084799850467,-8.492251512469188\r\n0bbf5e67,-130.42596535684112,0.16189164518456978\r\n12f0cd90,-323.38573216572394,40.57733768144723\r\n137d1e44,109.87577765568224,25.292305249698902\r\n14fee784,215.37664026645163,-28.55130817435117\r\n153552f5,162.0982422876511,15.43953448603704\r\n15e154a1,-174.66751555536135,4.429407485229716\r\n1619e2ca,-122.66197375240904,-1.1396303555610785\r\n1944513c,192.92218892062624,-35.727313380598346\r\n1a68190b,-179.87226741869182,24.34154693396812\r\n1b82b665,-180.21263645486377,24.199685340984164\r\n1bcb2fcf,134.67640477492176,7.301147847315222\r\n1d3fbf02,-65.42839762657745,-16.376699350183486\r\n1e5e3573,-127.56924939278667,-6.530143495371043\r\n203721ab,-110.33907313051265,-19.74019288855652\r\n20402d71,166.99870057527966,-23.645919891964663\r\n26623a50,273.86623631990904,-24.633333190131264\r\n26d3a96a,194.56976959504556,-9.021798191061912\r\n29b24409,199.02213676764876,-12.891711220841685\r\n29de15c8,106.04588242923441,-9.276190849467401\r\n2acc78d4,-155.0152670696581,20.36136376507802\r\n2be5c96a,-149.23069235430933,-4.942998794726274\r\n2c0c4a4e,-150.4060692108958,-6.992769905017548\r\n2d0c268a,128.3141591124132,19.29960006340326\r\n2d9f6cb9,246.41504582590963,-40.68357852342428\r\n2fa01aa6,-38.23204128719644,-29.360105373327706\r\n310c71a5,214.21189619207192,-15.933671611059477\r\n3407f5bf,-120.24353350178333,12.955906393196585\r\n3417285d,145.8462903484231,-9.66729947494128\r\n357c12f9,135.59826849297855,20.444315610713634\r\n35b30c7f,-256.7655516474922,-66.49672689894753\r\n35b3ef6a,223.58578560692575,-31.757888700323257\r\n38991fd4,216.96768874055235,-13.522548713657844\r\n3932faa6,120.5687766716982,46.03469368268595\r\n3a7dd95d,-61.955606931096995,17.187868746687865\r\n3df4b19e,189.91677909265016,-34.987702878708035\r\n3ef98781,131.89006365308757,-28.079381760528\r\n404c4384,49.998327369758016,-14.685497623914905\r\n43b06c70,179.49972912301655,51.224577963147425\r\n445af6c2,-140.5998959612312,-35.84865419743433\r\n44a94013,156.80860790891063,29.319210204569398\r\n460f859e,-64.34061516008414,21.426139285156165\r\n466fc788,141.8531433767152,29.588425508320515\r\n46d41ff6,-211.5767022211012,-50.72360674582514\r\n47ee27ab,204.00712217228735,17.940989275404192\r\n4a8ecc0b,121.58332859021098,46.04509259573508\r\n4c2208f5,107.27359500922668,-12.116894142264691\r\n4e050c92,-104.84639765085021,-9.103987191764464\r\n5138a660,-95.58576159735165,-13.28894668197415\r\n5397ceb1,-178.8555888858845,-14.985066876299996\r\n54753541,-106.68618108261984,-30.08498150667891\r\n5567d551,228.34086096967656,5.0104368665588375\r\n593876b6,-151.16866011493676,3.0324591435663697\r\n59f8b2e8,-171.0639240843741,14.268693582438544\r\n5beded95,-96.54686103854999,-8.187047864045004\r\n5cdd8dbc,87.51086910619422,28.679759283771304\r\n5f75f796,199.13460913977642,-18.74160201104734\r\n5fffa282,-134.41829835574208,16.363587408608264\r\n60e37807,76.74560534823429,31.985608143261842\r\n62569ae3,-82.45111350969374,-5.659039684244047\r\n6e6d122c,-148.04840203899008,-28.006767041600305\r\n6e9ccd38,251.7834087421476,-30.446915230226157\r\n71642c7d,-175.36132039179722,6.281136142985033\r\n722cf0d8,157.45718948826436,2.4195382784760437\r\n7271dd80,-129.6203710875253,-11.505194251268417\r\n729e9750,-158.53896279259882,-37.45054166339056\r\n73b1447d,-154.80509524732372,-46.737610252819714\r\n75cd5f11,-111.08561114475958,-32.86008541160612\r\n79507629,196.03973935021259,-12.373240283214685\r\n7acda2df,-105.93035391311122,19.373202980878574\r\n7b2c7b3c,77.07107396427118,24.371160921988533\r\n811f52b3,200.25096546445982,-17.470459351332558\r\n8255c867,39.34851415514394,30.14862505916282\r\n82aab6d2,260.2979984566995,-44.32607302590935\r\n861e71df,-104.85089355436756,-18.090509991678672\r\n877ed19c,-145.89891642826845,2.8074338121929365\r\n87aa3730,-160.49087417114688,-44.953138521086366\r\n89969c7a,157.94855084567646,36.30239790551786\r\n89f1085d,334.2752728943633,-79.55416631618574\r\n8bda1a11,-191.02000007462968,-19.206774713282975\r\n8cc800b3,-302.5126551135669,-13.672404543852629\r\n91b301ce,339.2422831166891,-54.09444309984091\r\n93209a3d,220.17299627789393,15.258948990208825\r\n940e709a,149.65015144193836,3.0842411744215905\r\n95b559e7,-75.75987925865215,11.411120987497252\r\n95c8427f,184.62455202446668,-5.3000892492585105\r\n98177ced,-115.48398346159303,12.56372911544949\r\n9a6b0392,-159.99679812392864,12.60016086564384\r\n9a95e33f,152.7923090193899,29.532476881983886\r\n9c8375f7,57.12563404296459,21.65081364718165\r\n9def2eb2,-170.85598024830097,-35.964576407359374\r\na056d01e,-213.98708776377364,20.89473058770083\r\na15bc102,124.8698804096046,-7.010315397197259\r\na3f71395,308.3408000025772,-26.25978449887927\r\na612d5a0,-192.65746692604992,24.598922088332234\r\na79fd2e7,150.83306464658125,16.82464325117615\r\na87433c9,-117.19155370464424,-51.596982455631654\r\na8ed028a,137.09350070072227,-32.46180616256331\r\naa95015b,-180.59217508374508,20.72653968573908\r\naaeffccb,-44.26186749980687,-21.12762960358125\r\nab3ced07,135.0546192324453,-12.06685527998269\r\nb32902c0,156.46368719091458,-22.590763253264576\r\nb32e4903,218.20512239593356,-25.49091258094778\r\nb5aa9634,257.4277735707234,-5.976347414377247\r\nbcebcc5f,-175.8550214507556,-10.062956243765566\r\nbe83e781,-97.40976789119739,-0.7411444090175261\r\nbeda8696,141.94100192920155,-15.497217620651668\r\nbf39bc20,-78.5069384118898,7.2224209388169\r\nbf70b410,207.07156582836376,-49.365470656013876\r\nc03d4aed,-184.04155642595975,12.600884971844803\r\nc2717d4f,174.59871270499121,-18.288857133789943\r\nc42fe306,156.79086370948997,-0.8308986657917741\r\nc59b6c4a,152.66048303763412,-6.809662143456949\r\nc6b782a4,223.43247655274516,-19.152005463907866\r\ncad13e8f,-89.93039965757646,-25.316493174300057\r\ncdc31d65,-142.00905101998814,21.273216180280006\r\nd41beaf2,207.80967758916478,-15.190822933505313\r\nd543916f,-169.4575767058927,2.357507028901876\r\nd5df0ff5,165.3571110129947,-8.990268818030675\r\nd6dd7330,155.25631242518418,7.939885448504913\r\nd7eb0be8,104.60853115386537,-4.680042290176225\r\nddc790ff,125.7352897802076,7.783905368446601\r\ne0f36d98,-125.05473544486003,-15.689796880208924\r\ne161bd0c,-79.81864384911009,-12.942890228758579\r\ne32da4ec,163.3473305474204,-19.083279289502485\r\ne45a2a62,139.4714768824821,26.952541552984506\r\ne5ff9fd2,59.49761365418179,12.926896219750631\r\ne730c32b,167.61808503802825,-7.648487987322651\r\ne737965a,223.58800533548583,-15.936483664751798\r\ne85074c5,-132.88381757313567,10.76069862461549\r\nea3a0e38,-250.09018637961864,2.2788748133336068\r\nebe69318,-122.24067768850051,-2.434287785539135\r\nec0d597b,-190.4728025023007,68.57359984912128\r\nee4aecac,-139.16299169093358,-57.91634312241562\r\nef04658e,152.48118177660592,-2.387973242424382\r\nefde6ac3,-174.1086072010134,11.066758612213778\r\nefe96181,-150.03002080341082,7.625751666218183\r\nf053a1b9,113.8454144641331,8.533786360951467\r\nf08774c3,105.55426639782866,35.05002820743212\r\nf140c2fa,155.86295646613632,-1.6264187694358212\r\nf16e25e3,147.1698426790563,-6.292567235091313\r\nf56d3ace,-136.00245855834578,-36.16014304654723\r\nf5d8aab6,165.25896773609253,-0.38966948374724497\r\nf631da6e,195.15642028528003,-33.102052782427975\r\nf6824bb0,242.20230420442272,-28.10056304571261\r\nf8afa78a,-187.9504668293635,-20.775822267016547\r\nf9fc81aa,-149.25820114066113,-23.771370623387735\r\nfb0904bd,130.61396815737055,6.854884389795725\r\nfd8f77fa,-186.04292475995032,3.350350977735566\r\nfde20ecf,-124.51065940433162,3.907598350838568\r\nffefef30,-155.20961021027136,10.405050827729323\r\n'

In [ ]:
from io import StringIO
from pathlib import Path
import json, os, time, warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

VERSION = "v1"
SMOKE = os.environ.get("ROGII_SMOKE", "0") == "1"
HORIZONTAL_SUFFIX = "__horizontal_well.csv"
TYPEWELL_SUFFIX = "__typewell.csv"
LEGAL_COLUMNS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
STRIDE = 10
TVT_STEP = 2.0
BAND_FT = 120.0
MAX_STEP = 4
CONFIGS = ["type_raw", "type_multiscale", "self_multiscale", "hybrid_multiscale"]


def find_root():
    roots = [
        Path(os.environ["ROGII_DATA"]) if os.environ.get("ROGII_DATA") else None,
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path.cwd() / "datasets", Path.cwd().parent / "datasets",
    ]
    roots.extend(parent / "datasets" for parent in list(Path.cwd().parents)[:4])
    for root in roots:
        if root is not None and (root / "train").exists() and (root / "test").exists():
            return root
    raise FileNotFoundError("ROGII dataset root not found")


ROOT = find_root()
TRAIN_DIR = ROOT / "train"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ridge_coefficients = pd.read_csv(StringIO(RIDGE_CSV)).set_index("well_id")
ids = sorted(path.name.removesuffix(HORIZONTAL_SUFFIX) for path in TRAIN_DIR.glob(f"*{HORIZONTAL_SUFFIX}"))
if SMOKE:
    ids = ids[:20]


def rmse(y_true, prediction):
    return float(np.sqrt(np.mean((np.asarray(y_true, float) - np.asarray(prediction, float)) ** 2)))


def robust_scale(values):
    values = np.asarray(values, float)
    median = np.nanmedian(values)
    scale = 1.4826 * np.nanmedian(np.abs(values - median))
    return float(max(scale, 1e-3))


print({"version": VERSION, "root": str(ROOT), "wells": len(ids), "smoke": SMOKE})


## Data - locally normalized sequence features

In [ ]:
def fill_gr(values):
    original = pd.Series(values, dtype=float)
    filled = original.interpolate(limit=100, limit_direction="both")
    fallback = float(original.median()) if original.notna().any() else 0.0
    return filled.fillna(fallback).to_numpy(float), original.notna().to_numpy(float)


def feature_bank(values):
    values = np.asarray(values, float)
    series = pd.Series(values)
    mean = series.rolling(61, center=True, min_periods=10).mean().bfill().ffill().to_numpy(float)
    std = series.rolling(61, center=True, min_periods=10).std().bfill().ffill().to_numpy(float)
    std = np.maximum(std, robust_scale(values) * 0.25)
    raw_z = np.clip((values - mean) / std, -5.0, 5.0)
    smooth_21 = series.rolling(21, center=True, min_periods=1).mean().to_numpy(float)
    smooth_61 = series.rolling(61, center=True, min_periods=1).mean().to_numpy(float)
    smooth_z = np.clip((smooth_21 - mean) / std, -5.0, 5.0)
    derivative = np.gradient(smooth_21)
    derivative = np.clip((derivative - np.median(derivative)) / robust_scale(derivative), -5.0, 5.0)
    slow_derivative = np.gradient(smooth_61)
    slow_derivative = np.clip((slow_derivative - np.median(slow_derivative)) / robust_scale(slow_derivative), -5.0, 5.0)
    return np.c_[raw_z, smooth_z, derivative, slow_derivative]


def reference_grid(typewell):
    typewell = typewell.dropna(subset=["TVT", "GR"]).sort_values("TVT")
    grouped = typewell.groupby("TVT", as_index=False)["GR"].median()
    lo = np.floor(grouped["TVT"].min() / TVT_STEP) * TVT_STEP
    hi = np.ceil(grouped["TVT"].max() / TVT_STEP) * TVT_STEP
    grid = np.arange(lo, hi + TVT_STEP, TVT_STEP)
    values = np.interp(grid, grouped["TVT"], grouped["GR"])
    return grid, feature_bank(values)


def self_reference(frame, reference_rows, grid):
    part = frame.iloc[reference_rows][["TVT_input", "GR"]].dropna()
    if len(part) < 30:
        return np.zeros((len(grid), 4)), np.zeros(len(grid), bool)
    bins = np.round(part["TVT_input"].to_numpy(float) / TVT_STEP) * TVT_STEP
    grouped = pd.DataFrame({"TVT": bins, "GR": part["GR"].to_numpy(float)}).groupby("TVT", as_index=False)["GR"].median()
    values = np.interp(grid, grouped["TVT"], grouped["GR"])
    coverage = (grid >= grouped["TVT"].min()) & (grid <= grouped["TVT"].max())
    return feature_bank(values), coverage


## Methods - monotonic slope-constrained DTW

In [ ]:
def emission_matrix(horizontal_features, sample_rows, reference_features, reliability, ridge, grid, channels):
    difference = horizontal_features[sample_rows][:, None, channels] - reference_features[:, channels][None, :, :]
    cost = np.minimum(difference * difference, 16.0).mean(axis=2)
    cost *= reliability[sample_rows, None]
    cost += 0.04 * ((grid[None, :] - ridge[:, None]) / 30.0) ** 2
    cost[np.abs(grid[None, :] - ridge[:, None]) > BAND_FT] = np.inf
    return cost


def decode_dtw(emission, grid, last_tvt, direction):
    ordered = emission if direction > 0 else emission[:, ::-1]
    ordered_grid = grid if direction > 0 else grid[::-1]
    rows, states = ordered.shape
    back = np.zeros((rows, states), np.int16)
    dp = ordered[0] + 2.0 * ((ordered_grid - last_tvt) / TVT_STEP) ** 2
    for row in range(1, rows):
        new = np.full(states, np.inf)
        source = np.zeros(states, np.int16)
        for step in range(MAX_STEP + 1):
            destination = np.arange(step, states)
            origin = destination - step
            candidate = dp[origin] + 0.08 * step * step
            better = candidate < new[destination]
            new[destination[better]] = candidate[better]
            source[destination[better]] = origin[better]
        dp = new + ordered[row]
        back[row] = source
    state = int(np.argmin(dp))
    path = np.empty(rows, int)
    path[-1] = state
    for row in range(rows - 1, 0, -1):
        state = int(back[row, state])
        path[row - 1] = state
    return ordered_grid[path], float(np.min(dp) / rows)


# Runnable self-check: the decoder recovers an exact increasing synthetic path.
synthetic_grid = np.arange(0.0, 20.0, 1.0)
synthetic_truth = np.arange(2, 10)
synthetic_cost = np.full((len(synthetic_truth), len(synthetic_grid)), 25.0)
synthetic_cost[np.arange(len(synthetic_truth)), synthetic_truth] = 0.0
synthetic_path, _ = decode_dtw(synthetic_cost, synthetic_grid, 2.0, 1)
assert np.array_equal(synthetic_path, synthetic_grid[synthetic_truth])


## Methods - prefix-only configuration and direction selection

In [ ]:
def ridge_path(frame, target_rows, anchor_row, anchor_tvt, well_id):
    md_values = frame["MD"].to_numpy(float)
    z = frame["Z"].to_numpy(float)
    span = max(float(md_values[target_rows[-1]] - md_values[anchor_row]), 1.0)
    x = (md_values[target_rows] - md_values[anchor_row]) / span
    base = anchor_tvt - (z[target_rows] - z[anchor_row])
    coefficient = ridge_coefficients.loc[well_id]
    return base + float(coefficient["ridge_c1"]) * x + float(coefficient["ridge_c2"]) * x * x


def all_emissions(frame, reference_rows, target_rows, ridge, grid, type_features):
    horizontal_gr, reliability = fill_gr(frame["GR"])
    horizontal_features = feature_bank(horizontal_gr)
    self_features, self_coverage = self_reference(frame, reference_rows, grid)
    sample_local = np.unique(np.r_[np.arange(0, len(target_rows), STRIDE), len(target_rows) - 1])
    sample_rows = target_rows[sample_local]
    type_raw = emission_matrix(horizontal_features, sample_rows, type_features, reliability, ridge[sample_local], grid, [0])
    type_multi = emission_matrix(horizontal_features, sample_rows, type_features, reliability, ridge[sample_local], grid, [0, 1, 2, 3])
    self_multi = emission_matrix(horizontal_features, sample_rows, self_features, reliability, ridge[sample_local], grid, [0, 1, 2, 3])
    self_multi[:, ~self_coverage] += 8.0
    hybrid = type_multi.copy()
    hybrid[:, self_coverage] = 0.5 * type_multi[:, self_coverage] + 0.5 * self_multi[:, self_coverage]
    return sample_local, {
        "type_raw": type_raw, "type_multiscale": type_multi,
        "self_multiscale": self_multi, "hybrid_multiscale": hybrid,
    }, float(1.0 - reliability[target_rows].mean()), float(self_coverage.mean())


def prefix_backtest(frame, well_id, grid, type_features):
    known_rows = np.flatnonzero(frame["TVT_input"].notna().to_numpy())
    cut = max(30, int(len(known_rows) * 0.70))
    reference_rows, validation_rows = known_rows[:cut], known_rows[cut:]
    if len(validation_rows) < 30:
        return {name: 1 for name in CONFIGS}, CONFIGS[0]
    anchor_row = int(reference_rows[-1])
    anchor_tvt = float(frame.loc[anchor_row, "TVT_input"])
    ridge = ridge_path(frame, validation_rows, anchor_row, anchor_tvt, well_id)
    sample_local, emissions, _, _ = all_emissions(frame, reference_rows, validation_rows, ridge, grid, type_features)
    result, direction_by_config = {}, {}
    truth = frame.loc[validation_rows, "TVT_input"].to_numpy(float)
    for name, emission in emissions.items():
        choices = []
        for direction in [-1, 1]:
            sampled, _ = decode_dtw(emission, grid, anchor_tvt, direction)
            prediction = np.interp(np.arange(len(validation_rows)), sample_local, sampled)
            choices.append((rmse(truth, prediction), direction))
        result[name], direction_by_config[name] = min(choices)
    return direction_by_config, min(result, key=result.get)


## Results - decode all wells

In [ ]:
prediction_frames, diagnostic_rows = [], []
start_time = time.time()

for index, well_id in enumerate(ids, 1):
    frame = pd.read_csv(TRAIN_DIR / f"{well_id}{HORIZONTAL_SUFFIX}", usecols=LEGAL_COLUMNS + ["TVT"])
    typewell = pd.read_csv(TRAIN_DIR / f"{well_id}{TYPEWELL_SUFFIX}", usecols=["TVT", "GR"])
    known_rows = np.flatnonzero(frame["TVT_input"].notna().to_numpy())
    target_rows = np.flatnonzero(frame["TVT_input"].isna().to_numpy())
    anchor_row = int(known_rows[-1])
    anchor_tvt = float(frame.loc[anchor_row, "TVT_input"])
    truth = frame.loc[target_rows, "TVT"].to_numpy(float)
    ridge = ridge_path(frame, target_rows, anchor_row, anchor_tvt, well_id)
    grid, type_features = reference_grid(typewell)
    directions, selected_config = prefix_backtest(frame, well_id, grid, type_features)
    sample_local, emissions, missing_share, self_coverage = all_emissions(
        frame, known_rows, target_rows, ridge, grid, type_features
    )
    predictions, path_costs = {"ridge_prior": ridge}, {}
    for name, emission in emissions.items():
        sampled, path_cost = decode_dtw(emission, grid, anchor_tvt, directions[name])
        predictions[name] = np.interp(np.arange(len(target_rows)), sample_local, sampled)
        path_costs[name] = path_cost
    predictions["prefix_selected"] = predictions[selected_config]
    oracle_config = min(CONFIGS, key=lambda name: rmse(truth, predictions[name]))
    predictions["oracle_config"] = predictions[oracle_config]
    prediction_frames.append(pd.DataFrame({
        "id": [f"{well_id}_{row}" for row in target_rows], "well_id": well_id,
        "row_index": target_rows, "target": truth, **predictions,
    }))
    correction = predictions["prefix_selected"] - ridge
    oracle_correction = truth - ridge
    diagnostic_rows.append({
        "well_id": well_id, "rows": len(target_rows), "ps_x": frame.loc[anchor_row, "X"], "ps_y": frame.loc[anchor_row, "Y"],
        "missing_gr_share": missing_share, "self_reference_coverage": self_coverage,
        "prefix_selected_config": selected_config, "oracle_config": oracle_config,
        "prefix_selected_direction": directions[selected_config],
        "correction_correlation": float(np.corrcoef(correction, oracle_correction)[0, 1]) if np.std(correction) > 0 else 0.0,
        "cycle_skip_share": float((np.abs(predictions["prefix_selected"] - truth) > 30.0).mean()),
        **{f"{name}_path_cost": path_costs[name] for name in CONFIGS},
    })
    if index % 100 == 0:
        print("decoded", index, "/", len(ids), "elapsed", round(time.time() - start_time, 1))

OOF = pd.concat(prediction_frames, ignore_index=True)
WELLS = pd.DataFrame(diagnostic_rows)
CANDIDATES = ["ridge_prior", *CONFIGS, "prefix_selected", "oracle_config"]
assert OOF["id"].is_unique and np.isfinite(OOF[CANDIDATES].to_numpy(float)).all()
print("OOF", OOF.shape)


## Results - pooled and spatial stability

In [ ]:
coordinates = StandardScaler().fit_transform(WELLS[["ps_x", "ps_y"]])
WELLS["spatial_fold"] = KMeans(n_clusters=min(5, len(WELLS)), random_state=42, n_init=10).fit_predict(coordinates)
fold_map = WELLS.set_index("well_id")["spatial_fold"]
OOF["spatial_fold"] = OOF["well_id"].map(fold_map)

SCORES = pd.DataFrame([
    {"candidate": candidate, "pooled_rmse": rmse(OOF["target"], OOF[candidate])}
    for candidate in CANDIDATES
]).sort_values("pooled_rmse").reset_index(drop=True)
SPATIAL_SCORES = pd.DataFrame([
    {"spatial_fold": fold, "candidate": candidate, "rows": len(group), "pooled_rmse": rmse(group["target"], group[candidate])}
    for fold, group in OOF.groupby("spatial_fold") for candidate in CANDIDATES
])
ridge_score = float(SCORES.set_index("candidate").loc["ridge_prior", "pooled_rmse"])
selected_score = float(SCORES.set_index("candidate").loc["prefix_selected", "pooled_rmse"])
spatial = SPATIAL_SCORES.pivot(index="spatial_fold", columns="candidate", values="pooled_rmse")
viable = bool((ridge_score - selected_score >= 1.0) and (spatial["prefix_selected"] < spatial["ridge_prior"]).all())

for candidate in CANDIDATES:
    per_well = OOF.groupby("well_id", sort=False).apply(
        lambda group: rmse(group["target"], group[candidate]), include_groups=False
    )
    WELLS[f"{candidate}_rmse"] = WELLS["well_id"].map(per_well)

display(SCORES)
display(spatial)
display(WELLS[["missing_gr_share", "self_reference_coverage", "correction_correlation", "cycle_skip_share"]].describe())
print({"viable": viable, "improvement_ft": ridge_score - selected_score})


## Takeaways and artifacts

In [ ]:
SCORES.to_csv(WORK / "alignment_lab_scores_v1.csv", index=False)
SPATIAL_SCORES.to_csv(WORK / "alignment_lab_spatial_scores_v1.csv", index=False)
WELLS.to_csv(WORK / "alignment_lab_wells_v1.csv", index=False)
prediction_path = WORK / "alignment_lab_oof_predictions_v1.parquet"
OOF.to_parquet(prediction_path, index=False)

summary = {
    "version": VERSION, "wells": int(OOF["well_id"].nunique()), "rows": int(len(OOF)),
    "stride": STRIDE, "tvt_step": TVT_STEP, "band_ft": BAND_FT, "max_step": MAX_STEP,
    "scores": SCORES.to_dict("records"),
    "ridge_prior_rmse": ridge_score, "prefix_selected_rmse": selected_score,
    "improvement_ft": ridge_score - selected_score,
    "spatial_fold_scores": SPATIAL_SCORES.to_dict("records"),
    "viability_rule": "prefix_selected improves Ridge by >=1 ft and wins in every spatial fold",
    "viable": viable,
    "decision": "promote_to_guarded_inference" if viable else "stop_or_revise_alignment",
    "prediction_artifact": prediction_path.name,
    "caveats": [
        "Spatial clusters are evaluation slices; this per-well decoder fits no cross-well model.",
        "Oracle configuration uses suffix TVT and is diagnostic only.",
        "Monotonic TVT cannot represent genuinely reversing local geology paths.",
        "No submission is created.",
    ],
}
(WORK / "alignment_lab_summary_v1.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
